# Results for model: meta/llama-3.1-8b-instruct

In [1]:
import polars as pl
import xgboost as xgb
import numpy as np
import pandas as pd

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Feature Engineering
df = df.with_columns([
    pl.col('feature_00').over(
        pl.Window(
            partition_by=['date_id'],
            order_by=['date_id'],
            frame='batch'
        )
    ).rank().over(
        pl.Window(
            partition_by=['date_id'],
            order_by=['date_id']
        )
    ).alias('feature_00_rank')
])

TOP_QUANTILE = df.filter(
    (pl.col('feature_00_rank') <= df['feature_00_rank'].quantile(0.85)) &
    (pl.col('date_id') == pl.col('date_id').sort('date_id').over(
        pl.Window(
            partition_by=['date_id'],
            order_by=['date_id']
        )
    ).rank().over(
        pl.Window(
            partition_by=['date_id'],
            order_by=['date_id']
        )
    ).alias('date_rank')
) | (
    (pl.col('feature_00_rank') <= df['feature_00_rank'].quantile(0.85)) &
    (pl.col('date_id') == pl.col('date_id').sort('date_id').over(
        pl.Window(
            partition_by=['date_id'],
            order_by=['date_id']
        )
    ).rank().over(
        pl.Window(
            partition_by=['date_id'],
            order_by=['date_id']
        )
    ).alias('date_rank')
).filter(pl.col('date_rank') <= 0.15 * (df['date_id'].sort('date_id').over(
    pl.Window(
        partition_by=['date_id'],
        order_by=['date_id']
    ).rank().over(
        pl.Window(
            partition_by=['date_id'],
            order_by=['date_id']
        ).alias('date_rank')
).alias('date_rank'))).alias('TOP_QUANTILE')

# Create new feature
df = df.with_columns([
    pl.when(pl.col('date_id').isin(df['date_id'].unique())).then(pl.col('feature_00')).otherwise(
        pl.when(pl.col('date_id') == pl.col('date_id').sort('date_id').over(
            pl.Window(
                partition_by=['date_id'],
                order_by=['date_id']
            )
        ).rank().over(
            pl.Window(
                partition_by=['date_id'],
                order_by=['date_id']
            )
        ).alias('date_rank')
        & (pl.col('date_rank') <= 0.15 * (df['date_id'].sort('date_id').over(
            pl.Window(
                partition_by=['date_id'],
                order_by=['date_id']
            ).rank().over(
                pl.Window(
                    partition_by=['date_id'],
                    order_by=['date_id']
                )
            ).alias('date_rank')
        ).alias('date_rank')
        ).then(pl.col('feature_00') * 0.8).otherwise(pl.col('feature_00') * 0.2).alias('feature_00')
    ).alias('feature_00')
])

# Split data
train_df = df.filter(~df['date_id'].isin(df['date_id'].unique()))
test_df = df.filter(df['date_id'].isin(df['date_id'].unique()))

# Train model
xgb_model = xgb.XGBRegressor()
xgb_model.fit(train_df[['feature_00', 'feature_01', 'feature_02', 'feature_03', 'feature_04', 'feature_05', 'feature_06', 'feature_07', 'feature_08', 'feature_09', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27', 'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32', 'feature_33', 'feature_34', 'feature_35', 'feature_36', 'feature_37', 'feature_38', 'feature_39', 'feature_40', 'feature_41', 'feature_42', 'feature_43', 'feature_44', 'feature_45', 'feature_46', 'feature_47', 'feature_48', 'feature_49', 'feature_50', 'feature_51', 'feature_52', 'feature_53', 'feature_54', 'feature_55', 'feature_56', 'feature_

SyntaxError: closing parenthesis ']' does not match opening parenthesis '(' on line 65 (2995565703.py, line 89)